In [1]:
import os
import numpy as np
import mediapy

import mujoco

from gaussian_renderer import GSRendererMuJoCo
from gs_playground import ROOT_PATH

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

mjcf_path = ROOT_PATH / "models" / "robots" / "manipulation" / "franka_emika_panda_robotiq" / "xmls" / "table30_01_press_three_buttons.xml"

_ASSETS_FRANKA_DIR = ROOT_PATH / "models" / "robots" / "manipulation" / "franka_emika_panda_robotiq"

gaussians = {
    "link1" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link1.ply").as_posix(),
    "link2" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link2.ply").as_posix(),
    "link3" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link3.ply").as_posix(),
    "link4" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link4.ply").as_posix(),
    "link5" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link5.ply").as_posix(),
    "link6" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link6.ply").as_posix(),
    "link7" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link7.ply").as_posix(),

    "robotiq_base"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "robotiq_base.ply").as_posix(),
    "left_driver"       : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_driver.ply").as_posix(),
    "left_coupler"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_coupler.ply").as_posix(),
    "left_spring_link"  : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_spring_link.ply").as_posix(),
    "left_follower"     : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_follower.ply").as_posix(),

    "right_driver"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_driver.ply").as_posix(),
    "right_coupler"     : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_coupler.ply").as_posix(),
    "right_spring_link" : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_spring_link.ply").as_posix(),
    "right_follower"    : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_follower.ply").as_posix(),

}
background = (_ASSETS_FRANKA_DIR / "3dgs" / "background.ply").as_posix()

Motphys profiler initialized: disabled


## Single Env

In [ ]:
# Make model, data, and renderer
mj_model = mujoco.MjModel.from_xml_path(mjcf_path.as_posix())
mj_data = mujoco.MjData(mj_model)
mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
mujoco.mj_forward(mj_model, mj_data)

gaussians.update({"background": background})
gsmx_renderer = GSRendererMuJoCo(gaussians, mj_model)
gaussians.pop("background")

gsmx_renderer.update_gaussians(mj_data)
results = gsmx_renderer.render(mj_model, mj_data, list(range(mj_model.ncam)), 320, 240)
for k, (rgb, depth) in results.items():
    rgb_np = rgb.cpu().numpy()
    mediapy.show_image(rgb_np)

## Batch Env

In [ ]:
from gaussian_renderer import BatchSplatConfig, MjxBatchSplatRenderer

import jax
import jax.numpy as jp
from mujoco import mjx

num_env = 4

# Make model, data, and renderer
mj_model = mujoco.MjModel.from_xml_path(mjcf_path.as_posix())
mj_data = mujoco.MjData(mj_model)
mujoco.mj_resetData(mj_model, mj_data)

cfg = BatchSplatConfig(body_gaussians=gaussians, background_ply=background, minibatch=32)
brenderer = MjxBatchSplatRenderer(cfg, mj_model)

mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)

# Create batch of environments (正确构造批量 mjx.Data)
def make_batch(data, n):
    return jax.tree.map(lambda x: jp.stack([x] * n, axis=0), data)

batch_data = make_batch(mjx_data, num_env)

# JIT compile the batched step function
jit_step = jax.jit(jax.vmap(mjx.step, in_axes=(None, 0)))

# Run one step to verify shapes
batch_data = jit_step(mjx_model, batch_data)
print(f"Batch xpos shape: {batch_data.xpos.shape}")  # (num_env, nbody, 3)
print(f"Batch xquat shape: {batch_data.xquat.shape}")  # (num_env, nbody, 4)
print(f"Batch cam_xpos shape: {batch_data.cam_xpos.shape}")  # (num_env, ncam, 3)
print(f"Batch cam_xmat shape: {batch_data.cam_xmat.shape}")  # (num_env, ncam, 3, 3)

# --- Step 2: batch_update_gaussians ---
body_pos = batch_data.xpos  # (Nenv, Nbody, 3)
body_quat = batch_data.xquat  # (Nenv, Nbody, 4, wxyz)
gsb = brenderer.batch_update_gaussians(body_pos, body_quat)
# --- Step 3: batch_env_render ---
cam_pos = batch_data.cam_xpos   # (Nenv, Ncam, 3)
cam_xmat = batch_data.cam_xmat  # (Nenv, Ncam, 3, 3)
H = 90; W = 120
fovy = np.array(mj_model.cam_fovy)[None, :]  # broadcast to (1, Ncam) or match cam count
rgb, depth = brenderer.batch_env_render(gsb, cam_pos, cam_xmat, H, W, fovy)

print('RGB:', rgb.shape, 'Depth:', depth.shape)

In [ ]:
def tile(img, d):
    assert img.shape[0] == d*d
    img = img.reshape((d,d)+img.shape[1:])
    return np.concat(np.concat(img, axis=1), axis=1)
mediapy.show_image(tile(rgb[:,0,...].cpu().numpy(), 2))

## Rollout

In [ ]:
# import time

# # Performance benchmark
# steps = 1000

# # Warmup
# batch_data = jit_step(mjx_model, batch_data)

# start = time.time()

# @jax.jit
# def rollout(state):
#     def step_fn(state, _):
#         return jit_step(mjx_model, state), None
#     final_state, _ = jax.lax.scan(step_fn, state, None, length=steps)
#     return final_state

# final_state = rollout(batch_data)
# _ = final_state.qpos.block_until_ready()

# end = time.time()
# fps = (num_env * steps) / (end - start)
# print(f"Simulated {num_env} environments for {steps} steps")
# print(f"FPS: {fps:,.2f}")